# Tool-Specific Reanalysis of Tully, Longoni, and Appel (2025) — Study 3

This notebook reproduces every numerical result and figure used in the paper
**"AI Receptivity or AI Adoption Breadth? A Tool-Specific Reanalysis of the Lower-Literacy / Higher-Usage Link"**.

It re-fits four model families on the publicly released Study 3 data of Tully, Longoni, and Appel (2025), under three
outcome groupings: pooled five-tool, text-AI only, and non-text AI only. Item-level models include task fixed effects.
All computations rely on the project's `tully_ai_literacy_robustness.py` library.

## 0. Setup

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.miscmodels.ordinal_model import OrderedModel

# Make the project library importable.
PROJECT_DIR = Path('.').resolve()
sys.path.insert(0, str(PROJECT_DIR))
from tully_ai_literacy_robustness import (
    build_long, fit_binary_logit, fit_multinomial_logit, fit_ols_average,
    fit_ordered_logit, make_exog, predicted_probabilities_ordered, read_table,
)

DATA_PATH = PROJECT_DIR / 'S3_data.xlsx'
FIG_DIR = PROJECT_DIR / 'figures'
FIG_DIR.mkdir(exist_ok=True)

LITERACY = 'SC0'
DEMOGRAPHIC_COVARS = ['Age', 'Income', 'GenKnow', 'Autonomy', 'Gender_dummy_1']
TABLE4_COVARS = ['TRI', 'GenKnow', 'Autonomy', 'Gender_dummy_1']
COVARS = DEMOGRAPHIC_COVARS
NONTEXT = ['AI_image', 'AI_productivity', 'AI_website', 'AI_healthapp']
ALL5 = NONTEXT + ['AI_text']

raw = read_table(DATA_PATH)
print('Shape:', raw.shape)
raw[['SC0', *ALL5, *DEMOGRAPHIC_COVARS, 'TRI']].describe().round(2)

## 1. Descriptive picture

Each AI tool is rated on a 1–5 frequency scale. The empirical distribution shows that non-text
categories are very right-skewed: most respondents have never used image generators, productivity tools,
website builders, or health apps. Text AI (ChatGPT-style) is the only category where a clear majority
of respondents are users.

In [ ]:
labels = {
    'AI_image': 'Image gen.\n(DALL-E)',
    'AI_productivity': 'Productivity\n(Zapier)',
    'AI_website': 'Web design\n(Canva)',
    'AI_healthapp': 'Health apps\n(Headspace)',
    'AI_text': 'Writing\n(ChatGPT)',
}
fig, ax = plt.subplots(figsize=(7.5, 4.0))
width = 0.16
x_cats = np.arange(1, 6)
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#9467bd', '#d62728']
for i, col in enumerate(ALL5):
    counts = raw[col].dropna().astype(int).value_counts().reindex(x_cats, fill_value=0)
    ax.bar(x_cats + (i - 2) * width, counts / counts.sum(), width=width,
           color=colors[i], label=labels[col])
ax.set_xticks(x_cats)
ax.set_xticklabels(['1 Never', '2 Once', '3 Occas.', '4 Freq.', '5 Weekly'])
ax.set_ylabel('Share of respondents')
ax.set_title('Reported usage frequency by AI tool (N=401)')
ax.legend(loc='upper right', fontsize=8)
ax.grid(True, axis='y', alpha=0.3)
fig.tight_layout()
fig.savefig(FIG_DIR / 'usage_distribution.pdf', dpi=300, bbox_inches='tight')
plt.show()

## 2. Block A — Pooled five-tool replication

We re-fit the original Study 3 specification using all five tool categories, with the same
covariates the paper reports: age, income, general knowledge, motivation for autonomy, and gender.
We obtain four estimates: OLS on participant-level averages, binary logit at the scale midpoint,
ordered logit (proportional odds), and a fully nominal multinomial logit. Item-level models include
task fixed effects.

In [ ]:
long_pooled = build_long(raw[raw['filter_$'] == 1].dropna(subset=COVARS), LITERACY, ALL5, id_col=None, covariates=COVARS)
id_col = '__row_id__'
fit_ols_average(long_pooled, id_col=id_col, literacy_col=LITERACY, covariates=COVARS)
fit_binary_logit(long_pooled, covariates=COVARS)
fit_ordered_logit(long_pooled, covariates=COVARS)
fit_multinomial_logit(long_pooled, covariates=COVARS)
predicted_probabilities_ordered(long_pooled, covariates=COVARS)

## 3. Block B — Text AI only

If the pooled effect is really about general AI receptivity, it should also appear when text AI
is examined on its own. We re-fit the four model families on the single `AI_text` outcome.

In [ ]:
long_text = build_long(raw[raw['filter_$'] == 1].dropna(subset=COVARS), LITERACY, ['AI_text'], id_col=None, covariates=COVARS)
fit_ols_average(long_text, id_col=id_col, literacy_col=LITERACY, covariates=COVARS)
fit_binary_logit(long_text, covariates=COVARS)
fit_ordered_logit(long_text, covariates=COVARS)
fit_multinomial_logit(long_text, covariates=COVARS)
predicted_probabilities_ordered(long_text, covariates=COVARS)

## 4. Block C — Non-text AI only (image, productivity, website, health app)

In [ ]:
long_nontext = build_long(raw[raw['filter_$'] == 1].dropna(subset=COVARS), LITERACY, NONTEXT, id_col=None, covariates=COVARS)
fit_ols_average(long_nontext, id_col=id_col, literacy_col=LITERACY, covariates=COVARS)
fit_binary_logit(long_nontext, covariates=COVARS)
fit_ordered_logit(long_nontext, covariates=COVARS)
fit_multinomial_logit(long_nontext, covariates=COVARS)
predicted_probabilities_ordered(long_nontext, covariates=COVARS)

## 5. Block D — Non-text AI, adoption threshold (y > 1)

Recoding the dependent variable as `1 = at least once / 0 = never` isolates the adoption margin
from the intensive-use margin.

In [ ]:
fit_binary_logit(long_nontext, covariates=COVARS, threshold=1)
fit_binary_logit(long_text, covariates=COVARS, threshold=1)

## 6. Predicted probability of non-use (Figure 1 of the paper)

From the ordered-logit fits we predict $\Pr(Y = 1)$ over a grid of standardized literacy values
for the text-only and non-text-only models. The slope is much steeper for non-text tools.

In [ ]:
grid = np.linspace(-2.0, 2.0, 17)
fig, ax = plt.subplots(figsize=(6.2, 4.2))
for label, cols, color, marker in [
    ('Text AI (writing assistants)', ['AI_text'], '#1f77b4', 'o'),
    ('Non-text AI tools', NONTEXT, '#d62728', 's'),
]:
    long = build_long(raw[raw['filter_$'] == 1].dropna(subset=COVARS), LITERACY, cols, id_col=None, covariates=COVARS)
    exog = make_exog(long, covariates=COVARS, task_fe=True)
    res = OrderedModel(long['y'].astype(int), exog, distr='logit').fit(
        method='bfgs', maxiter=1000, disp=False)
    base = pd.DataFrame(np.zeros((len(grid), exog.shape[1])), columns=exog.columns)
    base['z_literacy'] = grid
    pred = res.model.predict(res.params, exog=base)
    ax.plot(grid, pred[:, 0], marker=marker, color=color, linewidth=2,
            markersize=4, label=label)
ax.set_xlabel('AI literacy (standardized)')
ax.set_ylabel(r'Predicted $\Pr(\text{Never used})$')
ax.set_ylim(0.0, 1.0)
ax.grid(True, alpha=0.3)
ax.legend(loc='upper left')
fig.tight_layout()
fig.savefig(FIG_DIR / 'predicted_prob_never.pdf', dpi=300, bbox_inches='tight')
plt.show()

## 7. Coefficient summary (Figure 2 of the paper)

Side-by-side comparison of the AI-literacy coefficient by tool group and by model family.

In [ ]:
rows = []
for group_label, cols in [('Pooled (5 tools)', ALL5),
                          ('Non-text only', NONTEXT),
                          ('Text only', ['AI_text'])]:
    long = build_long(raw[raw['filter_$'] == 1].dropna(subset=COVARS), LITERACY, cols, id_col=None, covariates=COVARS)
    exog = make_exog(long, covariates=COVARS, task_fe=True)
    res = OrderedModel(long['y'].astype(int), exog, distr='logit').fit(
        method='bfgs', maxiter=1000, disp=False)
    rows.append({'group': group_label, 'model': 'Ordered logit',
                 'beta': float(res.params['z_literacy']),
                 'se': float(res.bse['z_literacy'])})
    for thr, model_label in [(3, 'Binary logit (y>3)'), (1, 'Binary adoption (y>1)')]:
        dat = long.copy(); dat['y_bin'] = (dat['y'] > thr).astype(int)
        ex = sm.add_constant(make_exog(dat, covariates=COVARS, task_fe=True),
                             has_constant='add')
        m = sm.GLM(dat['y_bin'], ex, family=sm.families.Binomial()).fit(cov_type='HC3')
        rows.append({'group': group_label, 'model': model_label,
                     'beta': float(m.params['z_literacy']),
                     'se': float(m.bse['z_literacy'])})
coef_df = pd.DataFrame(rows)
coef_df.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(7.0, 4.4))
groups = ['Pooled (5 tools)', 'Non-text only', 'Text only']
models = ['Ordered logit', 'Binary logit (y>3)', 'Binary adoption (y>1)']
colors = {'Ordered logit': '#1f77b4', 'Binary logit (y>3)': '#2ca02c',
          'Binary adoption (y>1)': '#d62728'}
y_positions = np.arange(len(groups))
offsets = {m: (i - 1) * 0.22 for i, m in enumerate(models)}
for model in models:
    sub = coef_df[coef_df['model'] == model]
    for _, row in sub.iterrows():
        ypos = y_positions[groups.index(row['group'])] + offsets[model]
        ci_lo, ci_hi = row['beta'] - 1.96 * row['se'], row['beta'] + 1.96 * row['se']
        ax.errorbar(row['beta'], ypos,
                    xerr=[[row['beta'] - ci_lo], [ci_hi - row['beta']]],
                    fmt='o', color=colors[model], capsize=3,
                    label=model if row['group'] == 'Pooled (5 tools)' else None)
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_yticks(y_positions); ax.set_yticklabels(groups)
ax.set_xlabel('Coefficient on +1 SD AI literacy (95% CI)')
ax.set_title('AI-literacy effect by tool group and model family')
ax.legend(loc='lower right'); ax.grid(True, axis='x', alpha=0.3)
fig.tight_layout()
fig.savefig(FIG_DIR / 'coefficient_comparison.pdf', dpi=300, bbox_inches='tight')
plt.show()

## 8. Summary

The pooled negative association replicates under every model family. Once the outcome is decomposed,
the AI-literacy slope is small and non-significant for text AI (ordered-logit $\beta=-0.097$, $p=.357$)
but strong for non-text AI ($\beta=-0.376$, $p<.001$). For non-text tools, the per-SD odds ratio of
ever having used the tool is $\approx 0.68$. Study 3 thus carries the most weight for **non-text AI adoption
breadth** rather than for general AI receptivity or text-AI usage intensity.

## 9. Original Table 4 covariate robustness


In [ ]:
rows = []
for spec_label, covars in [('Demographic-adjusted', DEMOGRAPHIC_COVARS),
                           ('Original Table 4 covariates', TABLE4_COVARS)]:
    for group_label, cols in [('Pooled (5 tools)', ALL5),
                              ('Text only', ['AI_text']),
                              ('Non-text only', NONTEXT)]:
        dat = raw[raw['filter_$'] == 1].dropna(subset=covars)
        long = build_long(dat, LITERACY, cols, id_col=None, covariates=covars)
        exog = make_exog(long, covariates=covars, task_fe=True)
        ordered = OrderedModel(long['y'].astype(int), exog, distr='logit').fit(
            method='bfgs', maxiter=1000, disp=False)
        rows.append({'spec': spec_label, 'group': group_label, 'model': 'Ordered logit',
                     'n_participants': long['__row_id__'].nunique(), 'n_rows': len(long),
                     'beta': float(ordered.params['z_literacy']),
                     'se': float(ordered.bse['z_literacy']),
                     'p': float(ordered.pvalues['z_literacy'])})
        for threshold, model_label in [(3, 'Binary logit (Y > 3)'), (1, 'Binary adoption (Y > 1)')]:
            binary_dat = long.copy()
            binary_dat['y_bin'] = (binary_dat['y'] > threshold).astype(int)
            binary_exog = sm.add_constant(make_exog(binary_dat, covariates=covars, task_fe=True),
                                          has_constant='add')
            binary = sm.GLM(binary_dat['y_bin'], binary_exog,
                            family=sm.families.Binomial()).fit(cov_type='HC3')
            rows.append({'spec': spec_label, 'group': group_label, 'model': model_label,
                         'n_participants': long['__row_id__'].nunique(), 'n_rows': len(long),
                         'beta': float(binary.params['z_literacy']),
                         'se': float(binary.bse['z_literacy']),
                         'p': float(binary.pvalues['z_literacy']),
                         'or': float(np.exp(binary.params['z_literacy']))})
robustness_df = pd.DataFrame(rows)
robustness_df.to_csv(PROJECT_DIR / 'result_table.csv', index=False)
robustness_df.round(3)
